# Design Your HGF Experiment

This notebook answers four questions you need answered **before collecting data**:

1. **Are my parameters recoverable?** Given my task, can the model reliably estimate individual differences?
2. **How many participants do I need?** What N gives 80% power to recover my target parameter?
3. **Can I distinguish groups?** If my treatment shifts omega_2 by *d*, can I detect it?
4. **Which model should I use?** 2-level vs 3-level — which does BMS prefer under my hypothesis?

Everything runs locally via **VB-Laplace** (no cluster needed).

### How the sweep works

For each combination of (N per group, effect size, iteration):

```
simulate cohort with group differences
  → fit 3-level HGF (VB-Laplace, ~5s)
  → fit 2-level HGF (VB-Laplace, ~3s)
  → parameter recovery (r, bias, RMSE)
  → model comparison (Laplace LME → BMS)
```

Repeat 10–20 times per cell, aggregate → power curves.

**Runtime:**
- Quick (3 N × 2 effects × 10 iter = 60 cells): ~5 min
- Standard (4 N × 3 effects × 20 iter = 240 cells): ~30 min
- First cell is slow (JAX JIT compilation); subsequent cells reuse the cache

### Related notebooks

- **`quickstart_hgf.ipynb`** — End-to-end pipeline tutorial: simulate, fit, recover, and compare models on a single cohort
- **`parameter_explorer.ipynb`** — Interactive slider dashboard to see how each HGF parameter affects belief trajectories and choice probabilities

In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["figure.dpi"] = 100

# ── Task selection ──────────────────────────────────────────────────
# Change this to switch tasks. The rest of the notebook adapts automatically.
TASK = "prl"  # "prl" for pick_best_cue, "patrl" for PAT-RL

### Choose your task config

By default this notebook uses the **pick_best_cue** (3-cue PRL) task.
To use a different task, change the `TASK` variable below. Available configs:

| Task | Config file | Description |
|------|-------------|-------------|
| `"prl"` | `configs/prl_analysis.yaml` | 3-cue partial-feedback PRL (default) |
| `"patrl"` | `configs/pat_rl.yaml` | Binary approach-avoid with ΔHR covariate |

The design sweep adapts automatically — different tasks have different
trial structures, parameter distributions, and session designs.

In [ ]:
from prl_hgf.env.task_config import load_config

config = load_config()

print("Group parameter distributions (baseline):")
print("=" * 55)
for group_name, gcfg in config.simulation.groups.items():
    print(f"\n  {group_name}:")
    print(f"    omega_2:  mean={gcfg.omega_2.mean:.1f}, sd={gcfg.omega_2.sd:.1f}")
    print(f"    omega_3:  mean={gcfg.omega_3.mean:.1f}, sd={gcfg.omega_3.sd:.1f}")
    print(f"    kappa:    mean={gcfg.kappa.mean:.2f}, sd={gcfg.kappa.sd:.2f}")
    print(f"    beta:     mean={gcfg.beta.mean:.1f}, sd={gcfg.beta.sd:.1f}")
    print(f"    zeta:     mean={gcfg.zeta.mean:.2f}, sd={gcfg.zeta.sd:.2f}")

print("\nSession deltas (treatment effects):")
print("=" * 55)
for group_name, sd in config.simulation.session_deltas.items():
    print(f"\n  {group_name}:")
    print(f"    omega_2_deltas: {sd.omega_2_deltas}")
    print(f"    kappa_deltas:   {sd.kappa_deltas}")

---
## Step 2: Run the Design Sweep

Choose your grid:
- **N per group**: sample sizes to test (e.g., 10, 20, 30, 50)
- **Effect sizes**: delta values to test (e.g., 0.3, 0.5, 0.7)
- **Iterations**: simulated datasets per cell (10 for quick, 20+ for stable estimates)

The sweep simulates a cohort, fits both HGF models with VB-Laplace,
computes parameter recovery, and runs model comparison — all locally.

In [ ]:
from prl_hgf.power.laplace_power import run_design_sweep, summarize_power

# Quick sweep — adjust for your needs
sweep_df = run_design_sweep(
    n_per_group_grid=[10, 20, 30],     # sample sizes to test
    effect_size_grid=[0.3, 0.7],       # effect sizes (omega_2 units)
    n_iterations=10,                   # iterations per cell (increase for stability)
    target_params=["omega_2", "beta", "zeta"],  # parameters to track
    bf_threshold=6.0,                  # BF threshold for evidence
    seed=42,
)

print(f"\nSweep complete: {len(sweep_df)} result rows")
sweep_df.head()

---
## Step 3: Are My Parameters Recoverable?

A parameter is **recoverable** if $r(\text{true}, \text{fitted}) \geq 0.7$.
Below that, individual differences are dominated by estimation noise —
you can't reliably rank participants.

We plot **mean recovery r** vs N for each parameter.

In [ ]:
summary_df = summarize_power(sweep_df)

params = sorted(summary_df["parameter"].unique())
n_params = len(params)

fig, axes = plt.subplots(1, n_params, figsize=(5 * n_params, 4), sharey=True)
if n_params == 1:
    axes = [axes]

for ax, param in zip(axes, params, strict=False):
    for es in sorted(summary_df["effect_size"].unique()):
        mask = (summary_df["parameter"] == param) & (summary_df["effect_size"] == es)
        sub = summary_df[mask].sort_values("n_per_group")
        ax.plot(sub["n_per_group"], sub["mean_r"], "o-", label=f"d = {es:.1f}")

    ax.axhline(0.70, color="red", ls="--", alpha=0.5, label="r = 0.7 threshold")
    ax.set_xlabel("N per group")
    ax.set_title(param)
    ax.legend(fontsize=8)

axes[0].set_ylabel("Mean recovery r")
fig.suptitle("Parameter Recovery vs Sample Size", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Step 4: What N Do I Need?

The **power curve** shows the probability that recovery $r \geq 0.7$
at each sample size. We want the curve to cross 80% — that's the
minimum N for reliable recovery.

In [ ]:
fig, axes = plt.subplots(1, n_params, figsize=(5 * n_params, 4), sharey=True)
if n_params == 1:
    axes = [axes]

for ax, param in zip(axes, params, strict=False):
    for es in sorted(summary_df["effect_size"].unique()):
        mask = (summary_df["parameter"] == param) & (summary_df["effect_size"] == es)
        sub = summary_df[mask].sort_values("n_per_group")
        ax.plot(sub["n_per_group"], sub["p_recoverable"], "o-", label=f"d = {es:.1f}")

    ax.axhline(0.80, color="gray", ls="--", alpha=0.5, label="80% power")
    ax.set_xlabel("N per group")
    ax.set_title(param)
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=8)

axes[0].set_ylabel("P(recovery r ≥ 0.7)")
fig.suptitle("Recovery Power Curves", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Print the recommendation table
print("Minimum N for 80% recovery power:")
print("=" * 50)

for param in params:
    param_df = summary_df[summary_df["parameter"] == param]
    for es in sorted(param_df["effect_size"].unique()):
        es_df = param_df[param_df["effect_size"] == es].sort_values("n_per_group")
        passing = es_df[es_df["p_recoverable"] >= 0.80]
        min_n = int(passing["n_per_group"].iloc[0]) if len(passing) > 0 else None
        min_n_str = f"N ≥ {min_n}" if min_n else "> max tested"
        print(f"  {param:>10s}  d={es:.1f}  →  {min_n_str}")

print("=" * 50)

---
## Step 5: Which Model Should I Use?

At each grid point we also run **Bayesian Model Selection** (Rigoux et al. 2014)
comparing the 2-level and 3-level HGF. The data is always generated from a
3-level process, so we expect the 3-level model to win — but only if N is
large enough.

The `bms_xp_3level` column gives the exceedance probability that the 3-level
model is better. Values near 1.0 = strong preference.

In [ ]:
# BMS results: mean exceedance probability for 3-level model
bms_df = (
    sweep_df.groupby(["n_per_group", "effect_size"])["bms_xp_3level"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 4))
for es in sorted(bms_df["effect_size"].unique()):
    sub = bms_df[bms_df["effect_size"] == es].sort_values("n_per_group")
    ax.plot(sub["n_per_group"], sub["mean"], "o-", label=f"d = {es:.1f}")
    ax.fill_between(
        sub["n_per_group"],
        sub["mean"] - sub["std"],
        sub["mean"] + sub["std"],
        alpha=0.15,
    )

ax.axhline(0.95, color="green", ls="--", alpha=0.5, label="Strong preference")
ax.axhline(0.50, color="gray", ls=":", alpha=0.5, label="Chance")
ax.set_xlabel("N per group")
ax.set_ylabel("Mean exceedance probability (3-level)")
ax.set_title("Model Comparison: Can BMS distinguish 3-level from 2-level?")
ax.set_ylim(-0.05, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

---
## Step 6: Encoding Your Own Hypothesis

To test a different hypothesis, edit `configs/prl_analysis.yaml`:

### Change group baseline differences
```yaml
simulation:
  groups:
    treatment:
      omega_2: {mean: -4.0, sd: 1.0}   # your treatment group baseline
    control:
      omega_2: {mean: -3.8, sd: 1.0}   # your control group baseline
```

### Change session effects (treatment shifts)
```yaml
  session_deltas:
    treatment:
      omega_2_deltas: [1.5, 0.8]   # drug effect at post_dose, followup
    control:
      omega_2_deltas: [0.3, 0.1]   # practice effect only
```

### Change the power sweep grid
```yaml
power:
  n_per_group_grid: [10, 15, 20, 25, 30, 40, 50]
  effect_size_grid: [0.3, 0.5, 0.7]
  n_iterations: 200
```

Then re-run the sweep above with `load_config()` — it picks up the new YAML automatically.

### What `effect_size_delta` does

The sweep's `effect_size_grid` adds $\delta$ to the treatment group's session deltas:

$$\Delta\omega_2^{\text{treatment}} = \Delta\omega_2^{\text{control}} + \delta$$

So if the control has `omega_2_deltas = [0.3, 0.1]` and $\delta = 0.5$,
the treatment gets `omega_2_deltas = [0.8, 0.6]`.
The **difference-in-differences = $\delta$** exactly.

### Testing effects on other parameters

The current sweep targets `omega_2` for the DiD contrast (the most
identifiable parameter). To test effects on `kappa` or `beta`, modify the
`session_deltas` in the YAML and pass `target_params=["kappa"]` to
`run_design_sweep()`.

---
## Step 7: Inspect Confounds

High correlations between fitted parameters ($|r| > 0.8$) mean
those parameters are **confounded** — you can't tell them apart.
This is especially common for $\omega_3$ and $\kappa$ in the
3-level HGF.

Let's check the inter-parameter confound matrix from one representative run.

In [ ]:
from prl_hgf.simulation.batch import simulate_batch
from prl_hgf.fitting.fit_vb_laplace_prl import fit_vb_laplace_prl, idata_to_fit_df
from prl_hgf.analysis.recovery import (
    build_recovery_df,
    compute_correlation_matrix,
    compute_recovery_metrics,
)
from prl_hgf.analysis.plots import plot_correlation_matrix

# Simulate a moderate cohort for a clean confound check
sim_df = simulate_batch(config)
sim_df = sim_df[sim_df["session"] == "baseline"].copy()
pids = sorted(sim_df["participant_id"].unique())[:30]
sim_df = sim_df[sim_df["participant_id"].isin(pids)].copy()

idata = fit_vb_laplace_prl(sim_df, model_name="hgf_3level", n_pseudo_draws=500)
fit_df = idata_to_fit_df(idata, ["omega_2", "beta", "zeta", "omega_3"])
recovery_df = build_recovery_df(sim_df, fit_df, min_n=0)

corr_df = compute_correlation_matrix(recovery_df)
fig = plot_correlation_matrix(corr_df)
plt.tight_layout()
plt.show()

print("\nInter-parameter correlations (|r| > 0.8 = potential confound):")
print(corr_df.round(3).to_string())

---
## Summary & Checklist

Before collecting data, verify:

- [ ] **Recovery**: target parameters achieve $r \geq 0.7$ at your planned N
- [ ] **Power**: $P(r \geq 0.7) \geq 80\%$ at your planned N and expected effect size
- [ ] **Confounds**: no inter-parameter $|r| > 0.8$ among your target parameters
- [ ] **BMS**: model comparison distinguishes 3-level from 2-level at your N
- [ ] **omega_3 caveat**: do NOT plan primary hypotheses on $\omega_3$ (poor recovery)

### MATLAB equivalence

| Design question | TAPAS | This toolbox |
|----------------|-------|-------------|
| Recoverability | Manual `tapas_simModel` loop | `run_design_sweep()` |
| Power analysis | Not built-in | `summarize_power()` + power curves |
| Model comparison | `spm_BMS` on single dataset | `compare_models_laplace()` per grid cell |
| Confound check | Manual correlation | `compute_correlation_matrix()` |

### Running the full sweep

For publication-quality power curves (200 iterations), run:
```bash
python scripts/design_experiment.py --n-grid 10 15 20 25 30 40 50 \
    --effect-grid 0.3 0.5 0.7 --n-iter 200
```

Or on the cluster for speed:
```bash
sbatch cluster/37_smoke_laplace_demo.slurm
```